# Build Your Agent
### IEEE SMC · MIT Bengaluru · Technical Symposium · 5 Sep 2026

You just spent an hour watching agent patterns run inside a browser demo — a scripted trace, pre-written, always ending the same way.

This notebook builds four of the eight patterns live — the ones that make a complete, self-contained story in ten minutes — and gives you the other four as a take-home extension in Section 6. The model is **real** (Google Gemini, free tier) and the failures are **real** — if your tool is described badly, it *will* misfire, live, in front of you. That's the point.

Every section below maps directly onto a stage you already saw in **Campus Agent Lab**, or a field you already filled in **Life Agent Builder**. Nothing here is new material — it's the same architecture, now wired to something that actually thinks.

**What you need:** a free Google account and five minutes. No GPU, no billing, no install beyond one `pip` line.

| Notebook section | Mirrors |
|---|---|
| 1 — Plain call | Campus Agent Lab · Stage 01 (Basic LLM) |
| 2 — Give it tools | Campus Agent Lab · Stage 03 (Tool-Calling) |
| 3 — Catch it lying | Campus Agent Lab · Stage 07 (Evaluator) |
| 4 — Add a rail | Campus Agent Lab · Stage 08 (Human-in-Loop) |
| 5 — Capstone | Life Agent Builder · Sense / Decide / Act / Rail |
| 6 — Take it further | Campus Agent Lab · Stages 02, 04, 05, 06 (Router, Planner, Multi-Agent, Memory) |

## Step 0 — Get a free API key (2 minutes, do this first)

1. Go to **[aistudio.google.com](https://aistudio.google.com/apikey)** and sign in with any Google account.
2. Click **Create API key**. Copy it — it's a long string starting with `AIza`.
3. This is free. No credit card. The free tier is tight, though — in a real run, the API reported limits as low as **5 requests** in a short window. Sections 1–4 (the required part) stayed well under that. Section 6, run in full, hit it. See the note further down for how this notebook handles it.
4. **Never commit this key to a public GitHub repo or paste it into chat.** If you're sharing this notebook publicly, use one of the two safe options in the next cell instead of pasting your key directly into the file.

**In Google Colab (recommended):** click the key icon (🔑) in the left sidebar → *Secrets* → add a secret named `GEMINI_API_KEY` → paste your key as the value → toggle *Notebook access* on. The code below will find it automatically.

**Running locally / no Colab secrets:** the fallback cell below will just ask you to paste the key when you run it — it is not saved anywhere.

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os

api_key = None

# Try Colab secrets first (recommended, keeps the key out of the notebook file)
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

# Fallback: ask for it interactively (works outside Colab too, not saved to disk)
if not api_key:
    import getpass
    api_key = getpass.getpass("Paste your Gemini API key (input hidden): ")

os.environ["GEMINI_API_KEY"] = api_key
print("Key loaded." if api_key else "No key found — go back to Step 0.")

In [ ]:
from google import genai

client = genai.Client()

# Model names on the free tier move fast. This is current as of the symposium —
# if it 404s for you, run the cell after the next one to see what's actually
# live right now and swap the string here.
MODEL = "gemini-3.8-flash"

**A note on the free tier — confirmed live, not theoretical:** running the required sections (1–4) plus the Router (6a) used about 15 requests without issue. Going on to Planner (6b) and Memory (6d) pushed the total past 20 and hit a real `429`. Rather than guess a magic safe number — the limit itself reported two different values (`20`, then `5`) in the same run — every call below goes through `call_model()`, which **rotates across several free-tier models** on a rate limit instead of just waiting. Falling over to a different model is usually instant; if every model is limited at once, it waits ~20s and tries the round again. If you still see red text after that, wait about a minute and re-run the cell.

In [20]:
import re, time, json

MODEL_CANDIDATES = [MODEL, "gemini-3.5-flash-lite", "gemini-2.5-flash-lite", "gemini-2.5-flash"]

def call_model(**kwargs):
    """Rotates across several free-tier models on a 429 instead of just waiting.
    Each model name has its OWN separate quota bucket, so falling over to a
    different model is usually instant — waiting on one exhausted model can
    take 20-50s. We confirmed live that the free tier gives 5-20 requests
    before a 429, and that number moves around — so this doesn't try to guess
    a magic number, it just tries other doors.
    """
    kwargs.pop("model", None)
    last_err = None
    for round_num in range(2):
        for model_name in MODEL_CANDIDATES:
            try:
                result = client.interactions.create(model=model_name, **kwargs)
                if model_name != MODEL_CANDIDATES[0]:
                    print(f"  (rate-limited on the primary model — used fallback: {model_name})")
                return result
            except Exception as e:
                msg = str(e)
                is_rate_limit = "429" in msg or "quota" in msg.lower() or "too_many_requests" in msg.lower()
                if not is_rate_limit:
                    raise
                last_err = e
        wait = 20
        print(f"⏳ All {len(MODEL_CANDIDATES)} candidate models rate-limited — waiting {wait}s (round {round_num + 1}/2)...")
        time.sleep(wait)
    raise RuntimeError(f"Still rate-limited across every model after retries. Wait ~60s and re-run. Last error: {last_err}")


def run_tool_loop(input_data, tool_defs, previous_interaction_id=None, max_steps=5):
    """Runs the FULL function-calling round trip: sends input, executes any
    function_call steps locally, submits the results back to the model, and
    repeats until it produces a final text answer (or max_steps safety cap).

    Without this, `result.output_text` is empty after a tool call — the model
    is still waiting to hear what the function returned. Returns
    (final_interaction, executed_calls) where executed_calls is a list of
    {"name", "arguments", "result"} dicts, one per tool call actually made.
    """
    executed_calls = []
    current_input = input_data
    prev_id = previous_interaction_id

    for _ in range(max_steps):
        kwargs = {"input": current_input, "tools": tool_defs}
        if prev_id:
            kwargs["previous_interaction_id"] = prev_id
        result = call_model(**kwargs)

        fn_steps = [s for s in result.steps if s.type == "function_call"]
        if not fn_steps:
            return result, executed_calls

        response_items = []
        for step in fn_steps:
            fn_result = available_functions[step.name](**step.arguments)
            executed_calls.append({"name": step.name, "arguments": step.arguments, "result": fn_result})
            response_items.append({
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": [{"type": "text", "text": json.dumps(fn_result)}],
            })

        current_input = response_items
        prev_id = result.id

    return result, executed_calls  # hit the safety cap — return whatever we have

In [21]:
try:
    _test = call_model(model=MODEL, input="Reply with just the word OK.")
    print(f"✓ Connected. Model '{MODEL}' responded: {_test.output_text.strip()}")
except Exception as e:
    print(f"✗ '{MODEL}' didn't work ({e}).\nRun the next cell to list models that ARE available to your key, then update MODEL above and re-run this cell.")

✓ Connected. Model 'gemini-3.8-flash' responded: OK


In [ ]:
# Only run this if the connection test above failed.
# It lists every model your API key can actually use, right now.
for m in client.models.list():
    print(m.name)

---
## 1 — A plain call (mirrors Stage 01 · Basic LLM)

You saw this in the browser: a model with no tools, no access to your timetable, no access to anything real — just text in, text out.

Ask it something it genuinely cannot know, and watch what it does instead of saying "I don't know."

In [22]:
response = call_model(
    model=MODEL,
    input="What time is my Robotics Lab workshop this Saturday, and is there a scheduling conflict with anything else on my calendar?",
)
print(response.output_text)

I don't have access to your personal calendar, emails, or schedule. 

If you can paste the details of your Saturday schedule and the workshop info here, I’d be happy to check for any conflicts!


Look at what you got back — one of two things happened, and both are worth noticing:

- **It answered anyway, confidently and wrong.** That's the exact failure mode from the deck: *"your agent will not throw an exception, it will lie politely."* You just watched it happen live.
- **It correctly said it has no access to your calendar.** Good — that means this particular model is well-calibrated on an *obvious* boundary. But notice the case that matters more: nothing here stops it from confidently answering a question that only *sounds* like it needs your calendar. Try `"Is it generally fine to book workshops on weeknights?"` — a fuzzier question with no clean refusal — and see if it hedges as cleanly the second time.

Either way: fix the actual gap by giving it real data to work from, not by hoping it declines correctly every time.

---
## 2 — Give it tools (mirrors Stage 03 · Tool-Calling)

Same two toy functions the browser demo used: a timetable and a conflict checker. This time they're real Python, and the model actually calls them.

In [23]:
# ---- real (toy) data, same shape as the Campus Agent Lab demo ----
TIMETABLE = {
    "tue": [{"time": "19:00", "event": "Algorithm Lab"}],
    "sat": [],  # Saturday is free
}
WORKSHOPS = {
    "robotics lab": {"day": "sat", "time": "16:00"},
}

def get_timetable(day: str):
    """Returns the student's existing calendar events for a given day (mon/tue/.../sun)."""
    return {"day": day, "events": TIMETABLE.get(day.lower(), [])}

def check_conflict(day: str, time: str):
    """Checks whether a proposed day+time collides with an existing calendar event."""
    events = TIMETABLE.get(day.lower(), [])
    clash = next((e for e in events if e["time"] == time), None)
    return {"conflict": clash is not None, "clashing_event": clash}

available_functions = {
    "get_timetable": get_timetable,
    "check_conflict": check_conflict,
}

tools = [
    {
        "type": "function",
        "name": "get_timetable",
        "description": "Returns the student's existing calendar events for a given day.",
        "parameters": {
            "type": "object",
            "properties": {"day": {"type": "string", "description": "e.g. 'sat', 'tue'"}},
            "required": ["day"],
        },
    },
    {
        "type": "function",
        "name": "check_conflict",
        "description": "Checks whether a proposed day+time collides with an existing calendar event.",
        "parameters": {
            "type": "object",
            "properties": {
                "day": {"type": "string"},
                "time": {"type": "string", "description": "24h format, e.g. '19:00'"},
            },
            "required": ["day", "time"],
        },
    },
]

print("Tools defined: get_timetable, check_conflict")

Tools defined: get_timetable, check_conflict


In [24]:
stage3_result, stage3_calls = run_tool_loop(
    "I want to register for the Robotics Lab workshop on Saturday at 4pm. "
    "Check my calendar first and tell me if there's a conflict.",
    tools,
)

for c in stage3_calls:
    print(f"→ model called {c['name']}({c['arguments']}) → {c['result']}")

print("\nFinal answer:\n", stage3_result.output_text)

  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
→ model called check_conflict({'time': '16:00', 'day': 'sat'}) → {'conflict': False, 'clashing_event': None}

Final answer:
 There is no conflict in your schedule for Saturday at 4:00 PM. Your calendar is clear at that time.


Notice the model didn't already know your calendar — it **asked its tools** before answering. That's the whole jump from Stage 01 to Stage 03: an agent isn't a smarter prompt, it's a loop that calls out for real data instead of guessing.

Now try something meaner: ask about **Tuesday at 7pm** instead of Saturday — the exact clash from the deck's evaluator slide — and see whether it actually catches its own conflict, or just answers anyway.

In [25]:
stage3b_result, stage3b_calls = run_tool_loop(
    "I want to register for a workshop on Tuesday at 7pm. Check my calendar and tell me if that works.",
    tools,
)
for c in stage3b_calls:
    print(f"→ model called {c['name']}({c['arguments']}) → {c['result']}")
print("\nFinal answer:\n", stage3b_result.output_text)

  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
→ model called check_conflict({'day': 'tue', 'time': '19:00'}) → {'conflict': True, 'clashing_event': {'time': '19:00', 'event': 'Algorithm Lab'}}

Final answer:
 You have a conflict on Tuesday at 7:00 PM—you are currently scheduled for "Algorithm Lab".


Did it say yes anyway, or did it catch the clash? Either is a realistic outcome — that's exactly the point of the next section. **Having the right tool call is not the same as reasoning correctly about the result.** Don't trust the model to police itself. Add a check that does it for you, in code, every time.

---
## 3 — Catch it lying (mirrors Stage 07 · Evaluator)

This is the slide from the deck: *"the tool call succeeded, it booked the clash anyway."* An evaluator is not another model call — it's ordinary code that checks the answer against ground truth, and forces a retry if it fails.

In [26]:
def evaluate(day: str, time: str) -> dict:
    """Ground-truth check, independent of anything the model said."""
    check = check_conflict(day, time)
    if check["conflict"]:
        return {"pass": False, "reason": f"{day} {time} clashes with {check['clashing_event']['event']}"}
    return {"pass": True, "reason": "no conflict"}


def run_with_evaluator(day: str, time: str, max_retries: int = 1):
    """Note: on retry, we evaluate whatever NEW day/time the model actually
    proposes via its own check_conflict() call — not the original bad slot.
    Checking the original slot again would make retrying pointless: it would
    fail identically every time no matter what the model suggested instead.
    """
    request_text = f"I want to register for a workshop on {day} at {time}. Check my calendar and tell me if that works."
    prev_id = None
    proposed_day, proposed_time = day, time

    for attempt in range(max_retries + 1):
        result, calls = run_tool_loop(request_text, tools, previous_interaction_id=prev_id)

        for c in calls:
            if c["name"] == "check_conflict":
                proposed_day = c["arguments"].get("day", proposed_day)
                proposed_time = c["arguments"].get("time", proposed_time)

        check = evaluate(proposed_day, proposed_time)
        print(f"Attempt {attempt + 1}: model proposes {proposed_day} {proposed_time}")
        print(f"  Model said → {result.output_text.strip()[:150]}")
        print(f"  Evaluator: {'PASS' if check['pass'] else 'FAIL — ' + check['reason']}")

        if check["pass"] or attempt == max_retries:
            return proposed_day, proposed_time, check

        prev_id = result.id
        request_text = (
            f"That's wrong — {check['reason']}. Do not confirm that booking. "
            f"Call check_conflict again with a different day/time that has no conflict, then confirm the new one."
        )

    return proposed_day, proposed_time, check


final_day, final_time, final_check = run_with_evaluator("tue", "19:00")

  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
Attempt 1: model proposes tue 19:00
  Model said → That time doesn't work. You already have an "Algorithm Lab" scheduled for Tuesday at 19:00.
  Evaluator: FAIL — tue 19:00 clashes with Algorithm Lab
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
Attempt 2: model proposes tue 15:00
  Model said → I checked Tuesday at 15:00, and it is completely free (no conflicts). Would you like me to register you for the workshop at that time?
  Evaluator: PASS


This is the difference between *having* an evaluator and *wiring* one — the phrase from Field Report 03 in the deck, about the alarm that fired and nobody escalated it. Here, the evaluator's FAIL isn't a log line. It goes straight back into the next request and forces a different answer.

---
## 4 — Add a rail (mirrors Stage 08 · Human-in-the-Loop)

Read-only calls (`get_timetable`, `check_conflict`) ran automatically above — no harm in being wrong, you just re-run them. Registering a student is a **write**. Gate it on reversibility, not on how confident the model sounds.

In [27]:
def register_workshop(day: str, time: str, name: str):
    """WRITE action — this is stage 08's gated tool, not auto-executed."""
    print(f"\n>>> About to register '{name}' for {day} {time}.")
    approve = input(">>> Approve this booking? (y/n): ").strip().lower()
    if approve == "y":
        print(f">>> ✓ Registered {name} for {day} {time}.")
        return True
    print(">>> ✗ Cancelled — no booking made.")
    return False

if final_check["pass"]:
    register_workshop(final_day, final_time, "you")
else:
    print("Evaluator never passed — correctly refusing to even offer the registration step.")


>>> About to register 'you' for tue 15:00.
>>> Approve this booking? (y/n): y
>>> ✓ Registered you for tue 15:00.


That `input()` call is the entire pattern. Nothing fancier is needed — the point of Stage 08 was never that approval gates are technically hard, it's that people forget to put them in front of the one function that can't be undone.

---
## 5 — Capstone: build your own (mirrors Life Agent Builder)

Pick one pattern from your own life — the same list from the browser tool:

- **The 2am scroll** — screen time past a threshold at night
- **The 11pm cart** — a big purchase, late, unreviewed
- **The 9-tab explosion** — too many open browser tabs, no cleanup
- **"I'll reply properly later"** — a message read but never answered
- **The forgotten deadline** — a recurring task, no completion signal
- or bring your own

Fill in the four functions below — the same SENSE / DECIDE / ACT / RAIL fields from the canvas — except this time `decide()` makes a real call to Gemini.

In [28]:
def sense():
    """TODO — return a dict describing the situation you're detecting.
    Example for the 2am scroll: {"screen_time_min": 95, "hour": 1, "next_alarm": "07:00"}
    """
    return {"screen_time_min": 95, "hour": 1, "next_alarm": "07:00"}


def decide(state: dict) -> str:
    """TODO — describe your situation and ask Gemini what the agent should say/do.
    This is the only step that calls the model — sense() and act() are plain code,
    exactly like the real tool.
    """
    prompt = (
        f"Current state: {state}. "
        f"Write a short, kind, one-sentence nudge for someone still scrolling at 1am "
        f"with a 7am alarm. Do not lecture. Do not use guilt."
    )
    result = call_model(model=MODEL, input=prompt)
    return result.output_text.strip()


def act(message: str):
    """TODO — what does the agent actually DO with the decision?
    Keep it to a notification-style action, not a destructive one.
    """
    print(f"📱 Notification: {message}")


def rail(state: dict) -> bool:
    """TODO — the one thing this agent must NEVER do.
    Return False to block act() from running at all.
    Example: never fire more than once a night, never fire before 9pm.
    """
    return state["hour"] >= 21 or state["hour"] <= 5


# ---- the loop — same shape every agent in this notebook has used ----
state = sense()
if rail(state):
    message = decide(state)
    act(message)
else:
    print("Rail blocked this run — outside the allowed window.")

  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
📱 Notification: That 7 AM alarm will come around surprisingly fast, so maybe it's time to let your eyes rest and catch some sweet dreams.


That's a complete agent: **sense → decide (with a real model) → act, wrapped in a rail.** Change `sense()` and `rail()` for your own pattern, re-run, and you've built the thing the browser tool only let you diagram.

### Ship it
- **Colab:** File → Save a copy in Drive, or File → Download → `.ipynb`
- **GitHub:** upload the `.ipynb` file to any repo — GitHub renders notebooks natively, no setup needed
- Never commit a notebook with your API key typed directly into a cell. If you used the Colab-secrets method above, your key was never written into this file.

### Where this came from
- Campus Agent Lab — `mit-symposium-demo-2026.web.app/campus-agent-lab.html`
- Life Agent Builder — `mit-symposium-demo-2026.web.app/life-agent-builder.html`
- The bridge deck (case studies, the arithmetic, the eight-pattern map) — `mit-symposium-demo-2026.web.app/pradyoth-bridge-deck.html`

---
## 6 — Take it further: the other four stages

This notebook deliberately built four of the eight Campus Agent Lab stages — 01, 03, 07, 08 — because together they're a complete, self-contained story in about ten minutes: *fails on its own → gets tools → gets caught lying → gets gated before it can act.* That's a full arc, not a shortcut.

The other four (**02 Router, 04 Planner, 05 Multi-Agent, 06 Memory**) are just as real, but each one needs its own extra piece of scaffolding, and adding all eight would have doubled the length of a session that only had an hour. So they're homework, not omissions — every one below builds directly on the `client`, `MODEL`, `tools`, and `available_functions` already defined above. Nothing new to set up, just new code.

Do these in any order, after the session, on your own copy.

### 6a — Router (mirrors Stage 02)

Before Stage 03 ever runs, a real system usually has to decide **which** specialist even handles the request — Academic, Timetable, Events, or Admin. That's routing: classify first, act second.

**Your task:** finish `route()` so it returns one of the four categories below for any request text. Two honest ways to build it — pick whichever, both are real routing:
- **Cheap and deterministic:** keyword matching (`if "workshop" in text: return "events"`)
- **Flexible:** one small model call asking it to output *only* the category name

**Run it once and check the output against your own judgement.** In testing, the model correctly caught three of four but filed "When is the robotics exam?" under `timetable` instead of `academic` — a real, live misclassification, not a notebook bug. That's not a failure of this exercise, it's the actual reason Stage 02 alone was never going to be enough — it's exactly the kind of mistake an evaluator (Section 3) exists to catch downstream.

In [29]:
CATEGORIES = ["academic", "timetable", "events", "admin"]

def route(request_text: str) -> str:
    """TODO — return exactly one of CATEGORIES.
    Try the keyword approach first, then compare it against a model-based one.
    """
    prompt = (
        f"Classify this student request into exactly one word from {CATEGORIES}. "
        f"Reply with only that word, nothing else.\n\nRequest: {request_text}"
    )
    result = call_model(model=MODEL, input=prompt)
    guess = result.output_text.strip().lower()
    return guess if guess in CATEGORIES else "admin"  # fallback bucket


import time as _time

for test in [
    "When is the robotics exam?",
    "Is Tuesday 7pm free on my calendar?",
    "Any workshops this weekend?",
    "How do I reset my student portal password?",
]:
    print(f"{route(test):>10}  ←  {test}")
    _time.sleep(2)  # light spacing — four calls back-to-back is exactly what trips a tight quota

  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
  academic  ←  When is the robotics exam?
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
 timetable  ←  Is Tuesday 7pm free on my calendar?
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
    events  ←  Any workshops this weekend?
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
     admin  ←  How do I reset my student portal password?


### 6b — Planner (mirrors Stage 04)

Stage 03 let the model call one tool at a time, reactively. A planner asks for the **whole ordered list of steps up front** — which catches wrong-order bugs before anything runs (checking a conflict before you even know the workshop time, for instance).

**Your task:** get the model to return a plan as a Python list of tool-call steps, then execute that list yourself, in order, using `available_functions` from section 2. Ask for JSON specifically — it's far easier to execute a plan than to parse prose.

In [30]:
import json as _json

def make_plan(goal: str) -> list:
    """TODO — ask the model for an ordered plan as a JSON list of
    {"tool": "...", "args": {...}} steps, using only get_timetable / check_conflict.
    """
    prompt = (
        f"Goal: {goal}\n"
        f"Available tools: get_timetable(day), check_conflict(day, time).\n"
        f'Return ONLY a JSON list like [{{"tool": "get_timetable", "args": {{"day": "sat"}}}}], '
        f"no prose, no markdown fences."
    )
    result = call_model(model=MODEL, input=prompt)
    text = result.output_text.strip().strip("`")
    return _json.loads(text)


plan = make_plan("Check whether Saturday 4pm works for a new workshop before I suggest it.")
print("Plan:", plan)

for step in plan:
    fn = available_functions.get(step["tool"])
    if fn:
        print(f"→ executing {step['tool']}({step['args']}) = {fn(**step['args'])}")

  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
Plan: [{'tool': 'check_conflict', 'args': {'day': 'sat', 'time': '4pm'}}]
→ executing check_conflict({'day': 'sat', 'time': '4pm'}) = {'conflict': False, 'clashing_event': None}


### 6c — Multi-Agent (mirrors Stage 05)

One coordinator, two specialists, each with a narrower job than the generalist from section 2. This is also the stage from the deck's Hugging Face field report — the moment you let one agent's output feed into another agent's decision, that hand-off is a trust boundary you now own.

**Your task:** write two specialist functions and a coordinator that calls both and merges their answers. Keep each specialist's system context narrow — that narrowness is the entire value of splitting them apart.

In [31]:
def academic_specialist(question: str) -> str:
    """TODO — answer ONLY academic/exam questions. Refuse anything else."""
    prompt = (
        "You are the Academic specialist. Only answer questions about courses, "
        "exams, or study topics. If asked anything else, say so and stop.\n\n"
        f"Question: {question}"
    )
    return call_model(model=MODEL, input=prompt).output_text.strip()


def schedule_specialist(question: str) -> str:
    """TODO — answer ONLY timetable/scheduling questions, using the real tools."""
    result, _ = run_tool_loop(question, tools)
    return result.output_text.strip()


def coordinator(request_text: str) -> str:
    """TODO — a trust boundary, not just a merge. Notice this coordinator does NOT
    blindly forward one specialist's claims to the other as fact — each specialist
    gets the ORIGINAL user request, not a paraphrase from a peer agent.
    """
    academic_part = academic_specialist(request_text)
    schedule_part = schedule_specialist(request_text)
    return f"Academic: {academic_part}\n\nSchedule: {schedule_part}"


print(coordinator("What should I study for the robotics exam, and is Saturday 4pm free for a workshop?"))

  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)
Academic: As the Academic specialist, I can only answer questions about courses, exams, or study topics. Therefore, I can answer your question regarding what to study for the robotics exam, but I cannot answer your question about whether Saturday at 4pm is free for a workshop. 

For the robotics exam, you should focus on your course syllabus, specifically reviewing kinematics, dynamics, sensor integration, control systems, and any programming assignments or lab work completed this term. 

Since your second question is outside my academic scope, I must stop here.

Schedule: Saturday at 4:00 PM is completely free! 

As for what to study for the robotics exam, I don't have access to your syllabus or study materials. Would you like me to help you find some genera

### 6d — Memory (mirrors Stage 06)

Not a bigger context window (see the deck) — a small, explicit store of facts that gets **read back in** on the next call. This is genuinely the simplest of the four to build, which is exactly why it's a good one to start with if you only do one.

**Your task:** finish `remember()` and `recall_context()` so a preference stated in one call actually changes the answer in the next one — without the model being told again.

In [32]:
MEMORY = {}

def remember(key: str, value: str):
    """TODO — just store it. This is the whole pattern."""
    MEMORY[key] = value
    print(f"✓ stored: {key} = {value}")


def recall_context() -> str:
    """TODO — turn MEMORY into a short string to prepend to the next prompt."""
    if not MEMORY:
        return ""
    facts = "; ".join(f"{k}: {v}" for k, v in MEMORY.items())
    return f"Known preferences — {facts}. Use these without asking again.\n\n"


# Turn 1 — state a preference, nothing else
remember("pref_time", "afternoon or weekend only")

# Turn 2 — a LATER, separate call, with no mention of the preference in the text
prompt = recall_context() + "Find me a workshop time from: Mon 9am, Sat 4pm, Wed 8am."
result = call_model(model=MODEL, input=prompt)
print("\n" + result.output_text.strip())

✓ stored: pref_time = afternoon or weekend only
  (rate-limited on the primary model — used fallback: gemini-3.5-flash-lite)

Based on your preference for weekends, the best workshop time for you is **Sat 4pm**.


With all eight built, you have the full Campus Agent Lab — not as a diagram you clicked through, but as code you wrote and ran against a real model. That's the whole distance this symposium was trying to close.